# Milestone 2 — Member 2: phát hiện LaTeX root

Notebook này triển khai **Task 1: `find_root()`** cho source của **một version**.
Các bước `gather()` và `build_tree()` sẽ được bổ sung sau.

**Chạy trên Colab:** dùng runtime CPU, chạy các cell từ trên xuống. Cell đầu tiên
kết nối Google Drive; cell cấu hình bên dưới chọn paper và version cần thử.
Không cần clone repo hay cài thêm package để chạy parser.

Dữ liệu nhóm nằm trong `arxiv-project/data/raw/<paper_id>/<version>/`.
Ví dụ đã quan sát: `2609-37833/v1/` chứa `main.tex` và `references.bib`.
Notebook chỉ đọc dữ liệu, không chỉnh sửa source M1 và không chạy toàn bộ corpus.

## 1. Kết nối dữ liệu

Nếu `arxiv-project` đang ở **Được chia sẻ với tôi**, thêm lối tắt của thư mục
vào **Drive của tôi** trước. Đăng nhập đúng tài khoản có quyền truy cập.
Nếu lối tắt có tên hoặc vị trí khác, sửa `RAW_DIR` trong cell cấu hình.
Khi chạy local, cell mount sẽ được bỏ qua; hãy đặt `RAW_DIR` thành đường dẫn local.

In [ ]:
try:
    from google.colab import drive
except ImportError:
    print("Đang chạy ngoài Colab: bỏ qua mount Drive, cấu hình RAW_DIR local ở cell sau.")
else:
    drive.mount("/content/drive")

## 2. Chọn một version

Chỉ cần chỉnh ba giá trị dưới đây. `source_dir` phải trỏ trực tiếp vào thư mục
version chứa `.tex`, không trỏ vào toàn bộ `raw` hoặc paper chứa nhiều version.
Nếu không tìm thấy đường dẫn, dùng bảng **Files** của Colab để lấy **Copy path**.

In [ ]:
from pathlib import Path

RAW_DIR = Path("/content/drive/MyDrive/arxiv-project/data/raw")
PAPER_ID = "2609-37833"
VERSION = "v1"

source_dir = RAW_DIR / PAPER_ID / VERSION

if not source_dir.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục version: {source_dir}\n"
        "Kiểm tra mount Drive, lối tắt arxiv-project và giá trị RAW_DIR."
    )

print("Source directory:", source_dir)
print("\nNội dung trực tiếp (tối đa 30 mục):")
entries = sorted(source_dir.iterdir(), key=lambda path: path.name)
for path in entries[:30]:
    kind = "DIR" if path.is_dir() else "FILE"
    print(f"[{kind}] {path.name}")

tex_files = sorted(
    path for path in source_dir.rglob("*")
    if path.is_file() and path.suffix.lower() == ".tex"
)
print("\nSố file .tex:", len(tex_files))
for path in tex_files[:20]:
    print("-", path.relative_to(source_dir).as_posix())
if not tex_files:
    print("Chưa thấy .tex: kiểm tra thư mục, file nén hoặc version không có TeX.")

## 3. Hàm hỗ trợ

Mask comment và các môi trường verbatim phổ biến; đọc file với encoding fallback;
dựng graph `input/include` sơ bộ để đánh giá ứng viên root. Cell này chỉ định nghĩa hàm.

In [ ]:
from pathlib import Path
import re


_VERBATIM_ENVS = {
    "verbatim",
    "verbatim*",
    "Verbatim",
    "lstlisting",
    "minted",
    "comment",
}

_DOCUMENTCLASS_RE = re.compile(
    r"\\documentclass\b"
    r"\s*(?:\[[^\]]*\]\s*)?"
    r"\{([^{}]+)\}",
    flags=re.DOTALL,
)

_BEGIN_DOCUMENT_RE = re.compile(
    r"\\begin\s*\{\s*document\s*\}"
)

_END_DOCUMENT_RE = re.compile(
    r"\\end\s*\{\s*document\s*\}"
)

_INCLUDE_RE = re.compile(
    r"\\(?:input|include)\b"
    r"\s*(?:\{([^{}]*)\}|([^\s%{}]+))"
)


def _mask_non_code(text):
    chars = list(text)

    def blank(start, end):
        for pos in range(start, end):
            if chars[pos] not in "\r\n":
                chars[pos] = " "

    i = 0
    n = len(text)

    while i < n:
        if text[i] == "%":
            backslashes = 0
            j = i - 1

            while j >= 0 and text[j] == "\\":
                backslashes += 1
                j -= 1

            if backslashes % 2 == 0:
                end = text.find("\n", i)

                if end == -1:
                    end = n

                blank(i, end)
                i = end
                continue

        if text[i] == "\\":
            if i + 1 < n and not text[i + 1].isalpha():
                i += 2
                continue

            verb_match = re.match(r"\\verb\*?(?![A-Za-z])", text[i:])

            if verb_match:
                delimiter_pos = i + verb_match.end()

                if (
                    delimiter_pos < n
                    and not text[delimiter_pos].isspace()
                ):
                    delimiter = text[delimiter_pos]
                    end = text.find(delimiter, delimiter_pos + 1)
                    newline = text.find("\n", delimiter_pos + 1)

                    if end != -1 and (newline == -1 or end < newline):
                        blank(i, end + 1)
                        i = end + 1
                        continue

            # Bỏ qua các môi trường chứa literal text.
            env_match = re.match(
                r"\\begin\s*\{([^{}]+)\}",
                text[i:],
            )

            if env_match and env_match.group(1) in _VERBATIM_ENVS:
                env_name = env_match.group(1)
                end_pattern = re.compile(
                    r"\\end\s*\{" + re.escape(env_name) + r"\}"
                )

                end_match = end_pattern.search(
                    text,
                    i + env_match.end(),
                )

                end = end_match.end() if end_match else n
                blank(i, end)
                i = end
                continue

            i += 1

            while i < n and text[i].isalpha():
                i += 1

            continue

        i += 1

    return "".join(chars)


def _read_tex(path):
    raw = path.read_bytes()

    try:
        return raw.decode("utf-8-sig"), None
    except UnicodeDecodeError:
        return (
            raw.decode("latin-1"),
            "UTF-8 decoding failed; used Latin-1 fallback.",
        )


def _resolve_include(argument, current_file, source_dir, known_files):
    argument = argument.strip()

    if not argument or any(
        token in argument for token in ("\\", "#", "{", "}", "\x00")
    ):
        return None

    relative_path = Path(argument)

    if relative_path.is_absolute() or relative_path.drive:
        return None

    if not relative_path.suffix:
        relative_path = relative_path.with_suffix(".tex")

    for base_dir in (current_file.parent, source_dir):
        candidate = (base_dir / relative_path).resolve()

        if (
            candidate.is_relative_to(source_dir)
            and candidate in known_files
        ):
            return candidate

    return None


def _reachable_files(start, graph):
    visited = set()
    pending = [start]

    while pending:
        current = pending.pop()

        if current in visited:
            continue

        visited.add(current)
        pending.extend(graph[current] - visited)

    return visited

## 4. Tìm root

Ứng viên phải có `documentclass` ngoài comment/verbatim. Class `standalone` bị loại.
Khi có nhiều ứng viên, heuristic xét dấu hiệu document, quan hệ include, tên main/ms
và tên tài liệu phụ. `ambiguous=True` yêu cầu kiểm tra thủ công.

**Giới hạn:** chưa diễn giải điều kiện TeX, macro trong đường dẫn và `includeonly`.
Điểm số không phải xác suất. `status="ok"` chỉ nghĩa là đã chọn được ứng viên.

In [ ]:
def find_root(source_dir):
    source_dir = Path(source_dir).expanduser().resolve()

    if not source_dir.is_dir():
        raise NotADirectoryError(
            f"Không tìm thấy thư mục source: {source_dir}"
        )

    def relative_name(path):
        return path.relative_to(source_dir).as_posix()

    result = {
        "status": "no_root",
        "source_dir": str(source_dir),
        "root_path": None,
        "selection_reason": [],
        "ambiguous": False,
        "candidates": [],
        "tex_file_count": 0,
        "warnings": [],
    }

    tex_files = sorted({
        path.resolve()
        for path in source_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() == ".tex"
            and path.resolve().is_relative_to(source_dir)
        )
    })

    result["tex_file_count"] = len(tex_files)

    if not tex_files:
        result["warnings"].append("Không tìm thấy file .tex.")
        return result

    clean_sources = {}

    for path in tex_files:
        try:
            text, encoding_warning = _read_tex(path)
        except OSError as exc:
            result["warnings"].append(
                f"Không đọc được {relative_name(path)}: {exc}"
            )
            continue

        clean_sources[path] = _mask_non_code(text)

        if encoding_warning:
            result["warnings"].append(
                f"{relative_name(path)}: {encoding_warning}"
            )

    known_files = set(clean_sources)

    graph = {path: set() for path in known_files}
    incoming = {path: set() for path in known_files}

    for path, text in clean_sources.items():
        for match in _INCLUDE_RE.finditer(text):
            argument = (
                match.group(1)
                if match.group(1) is not None
                else match.group(2)
            )

            target = _resolve_include(
                argument,
                current_file=path,
                source_dir=source_dir,
                known_files=known_files,
            )

            if target is not None:
                graph[path].add(target)
                incoming[target].add(path)

    candidates = []

    for path, text in clean_sources.items():
        declarations = list(_DOCUMENTCLASS_RE.finditer(text))

        if not declarations:
            continue

        class_name = declarations[0].group(1).strip()
        excluded = class_name.casefold() == "standalone"

        reachable = _reachable_files(path, graph)

        has_begin_here = bool(_BEGIN_DOCUMENT_RE.search(text))
        has_begin_reachable = any(
            _BEGIN_DOCUMENT_RE.search(clean_sources[file])
            for file in reachable
        )
        has_end_reachable = any(
            _END_DOCUMENT_RE.search(clean_sources[file])
            for file in reachable
        )

        score = 0
        reasons = []

        if has_begin_here:
            score += 40
            reasons.append("+40: có begin{document} trong chính file")
        elif has_begin_reachable:
            score += 25
            reasons.append("+25: begin{document} nằm trong file reachable")
        else:
            reasons.append("Không tìm thấy begin{document} reachable")

        if has_end_reachable:
            score += 5
            reasons.append("+5: có end{document} reachable")

        other_parents = incoming[path] - {path}

        if not other_parents:
            score += 20
            reasons.append("+20: không bị file .tex khác include")

        stem = path.stem.casefold()

        if stem in {"main", "ms"}:
            score += 20
            reasons.append("+20: tên file main/ms")

        name_tokens = set(re.split(r"[^a-z0-9]+", stem))

        auxiliary_tokens = {
            "rebuttal",
            "response",
            "supp",
            "supplement",
            "supplementary",
            "standalone",
        }

        if name_tokens & auxiliary_tokens:
            score -= 40
            reasons.append("-40: tên file có dấu hiệu tài liệu phụ")

        reachable_children = len(reachable) - 1
        include_bonus = min(reachable_children, 5)

        if include_bonus:
            score += include_bonus
            reasons.append(
                f"+{include_bonus}: có file .tex reachable"
            )

        if excluded:
            reasons.append("Loại khỏi lựa chọn: class standalone")

        if len(declarations) > 1:
            result["warnings"].append(
                f"{relative_name(path)} có nhiều documentclass; "
                "cần kiểm tra thủ công."
            )

        candidates.append({
            "path": relative_name(path),
            "documentclass": class_name,
            "excluded": excluded,
            "score": score,
            "has_begin_document": has_begin_here,
            "has_reachable_begin_document": has_begin_reachable,
            "reachable_tex_count": len(reachable),
            "included_by": sorted(
                relative_name(parent)
                for parent in other_parents
            ),
            "reasons": reasons,
        })

    candidates.sort(
        key=lambda item: (
            item["excluded"],
            -item["score"],
            item["path"],
        )
    )

    result["candidates"] = candidates

    eligible = [
        candidate
        for candidate in candidates
        if not candidate["excluded"]
    ]

    if not eligible:
        result["warnings"].append(
            "Không có ứng viên documentclass hợp lệ "
            "sau khi loại standalone."
        )
        return result

    selected = eligible[0]

    result["status"] = "ok"
    result["root_path"] = selected["path"]
    result["selection_reason"] = selected["reasons"]

    if len(eligible) > 1:
        result["warnings"].append(
            f"Có {len(eligible)} ứng viên root; "
            "đã chọn bằng heuristic."
        )

        score_gap = selected["score"] - eligible[1]["score"]

        if score_gap < 15:
            result["ambiguous"] = True
            result["warnings"].append(
                "Các ứng viên dẫn đầu có điểm gần nhau; "
                "cần kiểm tra thủ công."
            )

    if not selected["has_reachable_begin_document"]:
        result["warnings"].append(
            "Root được chọn không dẫn tới begin{document} "
            "mà bộ quét hiện tại nhận diện được."
        )

    return result

## 5. Chạy và xem ứng viên

Với version trong ảnh, kỳ vọng chọn `main.tex` nếu khai báo documentclass hợp lệ.
Bảng bên dưới giữ cả ứng viên bị loại để tiện kiểm tra.

In [ ]:
from IPython.display import display

root_result = find_root(source_dir)

print("Paper:", PAPER_ID, "| Version:", VERSION)
print("Status:", root_result["status"])
print("Root:", root_result["root_path"])
print("Ambiguous:", root_result["ambiguous"])
print("Number of .tex files:", root_result["tex_file_count"])

print("\nSelection reasons:")
for reason in root_result["selection_reason"]:
    print("-", reason)

print("\nWarnings:")
if not root_result["warnings"]:
    print("(none)")
for warning in root_result["warnings"]:
    print("-", warning)

rows = [
    {key: item[key] for key in (
        "path", "documentclass", "excluded", "score",
        "has_reachable_begin_document", "reachable_tex_count", "included_by"
    )}
    for item in root_result["candidates"]
]
if rows:
    try:
        import pandas as pd
    except ImportError:
        for row in rows:
            print(row)
    else:
        display(pd.DataFrame(rows))

## 6. Kiểm tra root được chọn

Xem khai báo documentclass và phần đầu source để xác nhận đây là bài chính.
Không có root thì cell này chỉ thông báo, không đọc một đường dẫn rỗng.

In [ ]:
root_file = None

if root_result["status"] != "ok":
    print("Chưa chọn được root. Xem warnings và kiểm tra source của version.")
else:
    root_file = Path(root_result["source_dir"]) / root_result["root_path"]
    root_text, encoding_warning = _read_tex(root_file)
    masked_text = _mask_non_code(root_text)
    declaration = _DOCUMENTCLASS_RE.search(masked_text)

    print("Root file:", root_file)
    print("Documentclass:", declaration.group(0) if declaration else "Không thấy")
    print("begin{document} trong root:", bool(_BEGIN_DOCUMENT_RE.search(masked_text)))
    if encoding_warning:
        print("Encoding warning:", encoding_warning)
    print("\n--- Preview: tối đa 4000 ký tự đầu ---")
    print(root_text[:4000])

## Bước tiếp theo

- Đổi `PAPER_ID` / `VERSION`, rồi chạy lại từ cell cấu hình để thử version khác.
- Kiểm tra thêm version có nhiều `.tex`, rebuttal/supplementary hoặc standalone.
- Khi chuyển sang Task 2, dùng `source_dir` và `root_file` làm đầu vào cho `gather()`.
- Trước khi commit, xóa output dài hoặc đường dẫn cá nhân. Không đưa dữ liệu M1 vào repo.